In [10]:
from presidio_analyzer import AnalyzerEngine, Pattern,PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import json
from pprint import pprint

Detection decides what PII exists. Anonymization decides what to do with it.

In [15]:
text_to_anonymize = """
Contact John Smith at john.smith@example.com or (415) 555-0132.
Credit card: 4111 1111 1111 1111
Address: 1 Hacker Way, Menlo Park, CA, id 123456
""".strip()

text_to_anonymize

'Contact John Smith at john.smith@example.com or (415) 555-0132.\nCredit card: 4111 1111 1111 1111\nAddress: 1 Hacker Way, Menlo Park, CA, id 123456'

In [17]:
analyzer = AnalyzerEngine()

In [7]:
# Narrow analysis to a single entity type\n
phone_results = analyzer.analyze(text=text_to_anonymize, entities=["PHONE_NUMBER"], language="en")
print("PHONE_NUMBER results:")
pprint(phone_results)

PHONE_NUMBER results:
[type: PHONE_NUMBER, start: 48, end: 62, score: 0.4]


In [9]:
# Narrow analysis to a single entity type\n
phone_results = analyzer.analyze(text=text_to_anonymize, entities=["PHONE_NUMBER", "PERSON", "EMAIL_ADDRESS"], language="en")
print("PHONE_NUMBER results:")
pprint(phone_results)

PHONE_NUMBER results:
[type: EMAIL_ADDRESS, start: 22, end: 44, score: 1.0,
 type: PERSON, start: 8, end: 18, score: 0.85,
 type: PHONE_NUMBER, start: 48, end: 62, score: 0.4]


In [18]:
# Or analyze across default recognizers\n
all_results = analyzer.analyze(text=text_to_anonymize, language="en")
print("All results:")
pprint(all_results)

All results:
[type: EMAIL_ADDRESS, start: 22, end: 44, score: 1.0,
 type: CREDIT_CARD, start: 77, end: 96, score: 1.0,
 type: PERSON, start: 8, end: 18, score: 0.85,
 type: LOCATION, start: 120, end: 130, score: 0.85,
 type: DATE_TIME, start: 139, end: 145, score: 0.85,
 type: URL, start: 22, end: 29, score: 0.5,
 type: URL, start: 33, end: 44, score: 0.5,
 type: PHONE_NUMBER, start: 48, end: 62, score: 0.4,
 type: US_DRIVER_LICENSE, start: 139, end: 145, score: 0.01]


In [19]:
#Adding a Custom Entity
id_pattern = Pattern(name="id_pattern", regex="\d{6}", score=0.5)

custom_recognizer = PatternRecognizer(supported_entity="ID", patterns=[id_pattern])

analyzer.registry.add_recognizer(custom_recognizer)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\user\AppData\Local\Temp\ipykernel_25868\2449138210.py:2: SyntaxWarning: invalid escape sequence '\d'
  id_pattern = Pattern(name="id_pattern", regex="\d{6}", score=0.5)


In [ ]:
all_results = analyzer.analyze(text=text_to_anonymize, language="en")
all_results

# our custom entity id is been detected 

[type: EMAIL_ADDRESS, start: 22, end: 44, score: 1.0,
 type: CREDIT_CARD, start: 77, end: 96, score: 1.0,
 type: PERSON, start: 8, end: 18, score: 0.85,
 type: LOCATION, start: 120, end: 130, score: 0.85,
 type: DATE_TIME, start: 139, end: 145, score: 0.85,
 type: URL, start: 22, end: 29, score: 0.5,
 type: URL, start: 33, end: 44, score: 0.5,
 type: ID, start: 139, end: 145, score: 0.5,
 type: PHONE_NUMBER, start: 48, end: 62, score: 0.4,
 type: US_DRIVER_LICENSE, start: 139, end: 145, score: 0.01]

In [21]:
for res in all_results:
    print(text_to_anonymize[res.start:res.end], res.entity_type)

john.smith@example.com EMAIL_ADDRESS
4111 1111 1111 1111 CREDIT_CARD
John Smith PERSON
Menlo Park LOCATION
123456 DATE_TIME
john.sm URL
example.com URL
123456 ID
(415) 555-0132 PHONE_NUMBER
123456 US_DRIVER_LICENSE


In [22]:
#  Anonymizing Data
anonymizer = AnonymizerEngine()

In [23]:
anonymized_text = anonymizer.anonymize(text=text_to_anonymize, analyzer_results=all_results)

In [25]:
print(anonymized_text.text)

Contact <PERSON> at <EMAIL_ADDRESS> or <PHONE_NUMBER>.
Credit card: <CREDIT_CARD>
Address: 1 Hacker Way, <LOCATION>, CA, id <DATE_TIME>


In [33]:
#Custom Anonymization 
from presidio_anonymizer.entities import OperatorConfig

operators = {
"PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 13, "masking_char": "*","from_end":True}),
"DEFAULT": OperatorConfig("replace", {"new_value": "[ANO]"})
}

anonymized_result = anonymizer.anonymize(
text=text_to_anonymize,
analyzer_results=all_results,
operators=operators)

In [34]:
print(anonymized_result.text)

Contact [ANO] at [ANO] or (*************.
Credit card: [ANO]
Address: 1 Hacker Way, [ANO], CA, id [ANO]


| Operator        | Purpose                               |
| --------------- | ------------------------------------- |
| `replace`       | Replace with fixed value              |
| `mask`          | Hide characters                       |
| `redact`        | Remove completely                     |
| `hash`          | Deterministically transform the value |
| `encrypt`       | Protect the original value            |
| Custom operator | Application-specific transformation   |
